# 11_prediction_target_design

Goal: define a realistic predictive target before any RF/XGBoost work.

This notebook answers only three questions:

1. Does the current repo only contain single-period foreign-population data?
2. Can comparable multi-period municipal foreign-population data be assembled?
3. What is the most realistic first predictive target?

This notebook is intentionally an audit-and-design notebook.
It does **not** run Random Forest, XGBoost, train/test split, or forecasting.


## Rules for this notebook

- Keep `data_raw/` as canonical
- Do not redo completed extraction
- Preserve the notebook workflow
- Do not claim a future target unless comparable multi-period municipal data are actually confirmed
- Use `current_hotspot_class` only as a fallback pilot target if no genuine future target is currently feasible


In [ ]:
import re
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from tokyo_foreigners.paths import PROJECT_ROOT, DATA_RAW_DIR, OUTPUTS_DIR

project_root = PROJECT_ROOT
data_raw = DATA_RAW_DIR
outputs_dir = OUTPUTS_DIR

print("PROJECT_ROOT:", project_root)
print("DATA_RAW_DIR:", data_raw)
print("OUTPUTS_DIR:", outputs_dir)


## Inventory all files in `data_raw/`


In [ ]:
all_files = sorted([p for p in data_raw.rglob("*") if p.is_file()])

inventory = pd.DataFrame({
    "relative_path": [str(p.relative_to(project_root)) for p in all_files],
    "name": [p.name for p in all_files],
    "suffix": [p.suffix.lower() for p in all_files],
})

print("Number of files in data_raw:", len(inventory))
inventory.head(50)


## Filter files likely related to population / foreign-population / feature layers


In [ ]:
keywords = [
    "foreign", "population", "pop", "gaikoku", "外国",
    "feature", "tokyo_", "ratio", "dissolved", "merge"
]

candidate_files = inventory[
    inventory["name"].str.lower().apply(
        lambda s: any(k.lower() in s for k in keywords)
    )
].copy()

print("Candidate file count:", len(candidate_files))
candidate_files


## Extract year clues from file names


In [ ]:
def extract_years(text: str):
    return re.findall(r"(19\d{2}|20\d{2})", text)

candidate_files["years_in_name"] = candidate_files["name"].apply(extract_years)
candidate_files[["relative_path", "years_in_name"]]


## Inspect columns in candidate tabular / spatial files


In [ ]:
def inspect_columns(path: Path):
    try:
        suffix = path.suffix.lower()
        if suffix == ".csv":
            df = pd.read_csv(path, nrows=5)
            return list(df.columns)
        if suffix in [".geojson", ".gpkg", ".shp"]:
            gdf = gpd.read_file(path)
            return list(gdf.columns)
        return []
    except Exception as e:
        return [f"ERROR: {e}"]

column_scan = []
for rel in candidate_files["relative_path"]:
    abs_path = project_root / rel
    cols = inspect_columns(abs_path)
    column_scan.append({
        "relative_path": rel,
        "columns": cols,
    })

column_scan_df = pd.DataFrame(column_scan)
column_scan_df


## Focus on target-relevant columns


In [ ]:
target_keywords = [
    "foreign", "ratio", "population", "pop", "total",
    "density", "year", "date", "time", "growth", "hotspot", "lisa", "cluster"
]

def filter_interesting_columns(cols):
    out = []
    for c in cols:
        c_str = str(c).lower()
        if any(k in c_str for k in target_keywords):
            out.append(c)
    return out

column_scan_df["interesting_columns"] = column_scan_df["columns"].apply(filter_interesting_columns)
column_scan_df[["relative_path", "interesting_columns"]]


## Inspect key current analysis layers if present


In [ ]:
key_paths = [
    data_raw / "tokyo_pop_dissolved.geojson",
    data_raw / "tokyo_mgwr_ready.geojson",
    data_raw / "tokyo_features_v4_extended_prep.geojson",
    data_raw / "tokyo_features_v5_extended_with_density.geojson",
]

key_layer_info = []

for p in key_paths:
    row = {"path": str(p.relative_to(project_root)), "exists": p.exists()}
    if p.exists():
        gdf = gpd.read_file(p)
        row["shape"] = gdf.shape
        row["columns"] = list(gdf.columns)
    else:
        row["shape"] = None
        row["columns"] = []
    key_layer_info.append(row)

key_layer_info_df = pd.DataFrame(key_layer_info)
key_layer_info_df


## Summarize year clues from names and columns


In [ ]:
def extract_years_from_columns(cols):
    years = []
    for c in cols:
        years.extend(re.findall(r"(19\d{2}|20\d{2})", str(c)))
    return sorted(set(years))

column_scan_df["years_in_columns"] = column_scan_df["columns"].apply(extract_years_from_columns)

year_summary = column_scan_df[["relative_path", "years_in_columns"]].copy()
year_summary["years_in_name"] = candidate_files["years_in_name"].values
year_summary


## Heuristic assessment of multi-period feasibility


In [ ]:
years_from_names = sorted(set(y for ys in candidate_files["years_in_name"] for y in ys))
years_from_columns = sorted(set(y for ys in column_scan_df["years_in_columns"] for y in ys))
all_detected_years = sorted(set(years_from_names + years_from_columns))

heuristic_assessment = {
    "detected_years_from_names": years_from_names,
    "detected_years_from_columns": years_from_columns,
    "all_detected_years": all_detected_years,
    "multi_period_signal": len(all_detected_years) >= 2,
}

heuristic_assessment


## Manual assessment template


In [ ]:
assessment = {
    "Q1_single_period_only": "TO_FILL",
    "Q2_multi_period_feasible": "TO_FILL",
    "Q3_most_realistic_first_target": "TO_FILL",
    "recommended_target_options": [
        "future_foreign_ratio",
        "foreign_population_growth_rate",
        "hotspot_emergence",
        "current_hotspot_class (fallback only)",
    ],
    "notes": [
        "If only one comparable municipal foreign-population period exists in the current repo, do not pretend to define a genuine future forecast target.",
        "If at least two comparable municipal periods can be assembled cleanly, prefer future_foreign_ratio first.",
        "If the continuous target is unstable or the time interval is irregular, foreign_population_growth_rate is the second choice.",
        "If multi-period continuous targets are not feasible now, current_hotspot_class may be used only as a temporary pilot target, not as a genuine forecast target.",
    ],
}

assessment


## Suggested interpretation rule


In [ ]:
if heuristic_assessment["multi_period_signal"]:
    suggested_rule = {
        "Q1_single_period_only": "Probably no, but manual verification required",
        "Q2_multi_period_feasible": "Possibly yes, subject to same-unit comparability check",
        "Q3_most_realistic_first_target": "future_foreign_ratio (preferred) or foreign_population_growth_rate",
    }
else:
    suggested_rule = {
        "Q1_single_period_only": "Likely yes in the current repo state",
        "Q2_multi_period_feasible": "Not yet demonstrated",
        "Q3_most_realistic_first_target": "current_hotspot_class only as a temporary pilot target",
    }

suggested_rule


## Save audit outputs


In [ ]:
audit_dir = outputs_dir / "round_11_target_audit"
audit_dir.mkdir(parents=True, exist_ok=True)

inventory.to_csv(audit_dir / "data_raw_inventory.csv", index=False, encoding="utf-8-sig")
candidate_files.to_csv(audit_dir / "candidate_files.csv", index=False, encoding="utf-8-sig")
column_scan_df.to_csv(audit_dir / "candidate_file_columns.csv", index=False, encoding="utf-8-sig")
key_layer_info_df.to_csv(audit_dir / "key_layer_info.csv", index=False, encoding="utf-8-sig")
year_summary.to_csv(audit_dir / "year_summary.csv", index=False, encoding="utf-8-sig")

with open(audit_dir / "target_audit_summary.txt", "w", encoding="utf-8") as f:
    f.write("Heuristic assessment\n")
    f.write(str(heuristic_assessment) + "\n\n")
    f.write("Suggested rule\n")
    f.write(str(suggested_rule) + "\n\n")
    f.write("Manual assessment template\n")
    f.write(str(assessment) + "\n")

print("Saved audit outputs to:", audit_dir)


## How to conclude this notebook

After running the cells above, write a brief manual conclusion in the notebook itself:

- If the repo currently contains only one comparable municipal foreign-population period, keep predictive work at the audit stage.
- If multiple comparable municipal periods can be assembled cleanly, define `future_foreign_ratio` as the preferred target.
- Only use `current_hotspot_class` as a temporary pilot target when a genuine future target is not yet available.
